# 프로토콜 v1.2 — WS1: 개발 데이터 학습 곡선, TSMixer·TCN·S-pool (탐색 분석, GPU)

CPU 노트북이 만든 추출 목록(`ws1_draws.csv`, 해시 확인)과 **똑같은 조합**으로 CruiseBench(커밋 `ef26d40`) 모델을 학습하고, 빼 둔 개발 엔진의 순항 창을 예측합니다. 시험 데이터는 쓰지 않습니다.

- **S-target (주):** 대상 서브셋의 학습 m대만으로 TSMixer, TCN을 학습합니다. 415회씩이고, 시드 = 추출 번호 + 1입니다.
- **S-pool (보조):** TSMixer가 다른 8개 서브셋의 개발 엔진 전체와 대상 서브셋 2대(첫 조합)로 학습합니다. 평가 엔진 55대이고, 레이블은 각 창이 속한 서브셋의 η = 5% 상한입니다.
- **원 코드 그대로인 것:** 모델, 손실, 최적화(Adam 3e-4), 50 epoch, 배치 512, 창 256·간격 10, 순항 마스크, 원 코드의 읽기·마스크 함수
- **바뀐 것 (데이터 경로만):**
  1. 학습 엔진을 추출 목록으로 제한하고, 정규화(MinMax)를 학습 엔진에만 맞춥니다(원 코드는 dev 전체). 빼 둔 엔진의 정보가 학습에 새지 않게 하기 위해서입니다.
  2. 창을 미리 만들어 두지 않고 필요할 때 잘라 씁니다. 창의 내용과 순서는 원 코드와 같고, 점검 실행에서 원 `Data_Loader`의 창과 하나하나 같은지 확인합니다.
  3. S-pool은 여러 서브셋을 이어 붙입니다(원 코드에 없는 확장).
- **계산량 (v1.2 2.6):** 점검 실행에서 잰 학습 속도로 전체 시간을 추정합니다. 한도는 40 GPU시간(L4)이고, 넘으면 다음 순서로 줄입니다: ① S-pool 평가 엔진 55 → 27, ② S-target 추출 3 → 2, ③ m = 3 제외.

**순서 (v1.2 6절)**
1. `SMOKE = True`로 처음부터 실행합니다. 커밋 주소는 아직 필요 없습니다.
   - 점검 실행은 원 코드와 창이 같은지, 2 epoch 학습, 중단 후 이어서 실행, 채점 셀 작동을 확인하고 시간을 잽니다.
   - 성능 지표는 출력하지 않고, 결과 파일은 지웁니다.
2. 점검이 통과하면 이 노트북을 커밋하고, `SMOKE = False`와 커밋 주소를 넣은 뒤 다시 처음부터 실행합니다.

**런타임:** GPU L4, 고용량 RAM. 끊기면 다시 실행하면 끝난 학습은 건너뜁니다.

In [ ]:
# === 1. 설정 ===
import os, sys, json, glob, time, hashlib, subprocess, zipfile, re, shutil
import numpy as np, pandas as pd
SMOKE = True                     # 점검 실행이면 True. 통과 후 커밋하고 False로 바꿔 본 실행
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'; WORK = '/content/ncmapss_raw'; LOCAL = '/content/ws1gpu'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm'); WORK = os.environ.get('NCM_WORK', './ncmapss_raw'); LOCAL = os.environ.get('WS1G_LOCAL', './ws1gpu_local')
W1 = f'{ROOT}/addendum_v12/ws1'; OUT = f'{W1}/dl'
for d in [OUT, f'{OUT}/fits', WORK, LOCAL]: os.makedirs(d, exist_ok=True)
URL = r'https://github\.com/.+/commit/[0-9a-f]{7,40}$'
ADDENDUM_V12_URL = 'https://github.com/goniworld/pibdp-ncmapss/commit/f01eecf82f5bc45ed9a90d44d8361b8dcb217d63'
RUN_COMMIT_URL = ''              # 점검 통과 후 이 노트북을 커밋한 주소 (본 실행에만 필요)
if not SMOKE: assert re.match(URL, RUN_COMMIT_URL), '점검 통과 후 커밋하고 주소를 넣으세요.'
SUB9 = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008']
CAPS = dict(zip(SUB9, [38, 34, 34, 43, 38, 41, 39, 34, 28]))
BUDGET_H, CU_PER_H, EPOCHS = 40.0, 1.71, 50
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEV == 'cuda' else 'cpu'; print('장치:', GPU, '| 점검 실행' if SMOKE else '| 본 실행')

In [ ]:
# === 2. N-CMAPSS 내려받기 (Phase 2b와 같은 파일인지 해시로 확인) ===
URLZ = 'https://phm-datasets.s3.amazonaws.com/NASA/17.+Turbofan+Engine+Degradation+Simulation+Data+Set+2.zip'
MANUAL_ZIP = ''
def h5_list(): return sorted(glob.glob(os.path.join(WORK, '**', '*.h5'), recursive=True))
if not h5_list():
    zpath = os.path.join(WORK, 'ncmapss.zip')
    if MANUAL_ZIP: subprocess.run(['cp', MANUAL_ZIP, zpath])
    elif not (os.path.exists(zpath) and zipfile.is_zipfile(zpath)):
        subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', zpath, URLZ])
    done = set()
    while True:
        zs = [z for z in glob.glob(os.path.join(WORK, '**', '*.zip'), recursive=True) if z not in done]
        if not zs: break
        for z in zs:
            done.add(z)
            if zipfile.is_zipfile(z):
                with zipfile.ZipFile(z) as f: f.extractall(os.path.splitext(z)[0] + '_x')
    for z in glob.glob(os.path.join(WORK, '**', '*.zip'), recursive=True): os.remove(z)
H5 = {os.path.basename(p).replace('.h5', '').replace('N-CMAPSS_', ''): p for p in h5_list() if '/flat/' not in p}
REFH = json.load(open(f'{ROOT}/phase2b/sha256_h5.json'))
FLAT = os.path.join(WORK, 'flat'); os.makedirs(FLAT, exist_ok=True)
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''): h.update(b)
    return h.hexdigest()
for s in SUB9:
    assert os.path.getsize(H5[s]) == REFH[s]['bytes'] and sha(H5[s]) == REFH[s]['sha256'], f'{s}: Phase 2b와 다른 파일'
    dst = os.path.join(FLAT, f'N-CMAPSS_{s}.h5')
    if not os.path.exists(dst): os.symlink(os.path.abspath(H5[s]), dst)
print('h5 9개 해시 확인 완료')

In [ ]:
# === 3. CruiseBench 고정 커밋 + WS1 데이터 경로 코드 ===
CB = os.environ.get('CB_DIR') or os.path.join(LOCAL, 'CruiseBench')
if not os.path.exists(CB): subprocess.run(['git', 'clone', '-q', 'https://github.com/NostalgiaJohn/CruiseBench', CB], check=True)
subprocess.run(['git', '-C', CB, 'checkout', '-q', 'ef26d40'], check=True)
CB_HEAD = subprocess.run(['git', '-C', CB, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip(); assert CB_HEAD.startswith('ef26d40')
assert not subprocess.run(['git', '-C', CB, 'status', '--porcelain', '--untracked-files=no'], capture_output=True, text=True).stdout.strip(), '원 코드가 바뀌어 있음'
GLIB_SRC = '"""WS1 data-driven learning curves with CruiseBench models (protocol addendum v1.2, section 2).\nOnly the data pipeline is new: training engines are restricted to the WS1 draw, the scaler is fitted on them,\nwindows are served lazily (identical windows and order to CruiseBench _make_windows), and S-pool pools subsets.\nModels, loss, optimiser, epochs and batch sizes are the original CruiseBench code and defaults."""\nimport pathlib, time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn import preprocessing\nfrom torch.utils.data import Dataset, DataLoader\nimport data_load as cbload\n\nW, S = 256, 10\n\n\ndef load_dev(h5, mask):\n    """Original CruiseBench reading and cruise-mask filtering of the dev split (unscaled)."""\n    L = cbload.Data_Loader.__new__(cbload.Data_Loader)\n    L.data_path = pathlib.Path(h5); L.dataset_name = cbload._dataset_name_from_path(L.data_path)\n    L.cruise_mask_path = pathlib.Path(mask)\n    df, feats = L._read_raw_split(\'dev\')\n    df = L._filter_to_cruise_mask(df, split=\'dev\').reset_index(drop=True)\n    return df, feats\n\n\nclass Windows:\n    """Window index over rows of the given units; same windows and order as Data_Loader._make_windows."""\n    def __init__(self, parts, feats, scaler, fit_scaler):\n        # parts: list of (df_subset_rows, cap); rows keep the original order inside each part\n        dfs = [d.assign(_cap=cap, _part=i) for i, (d, cap) in enumerate(parts)]\n        df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0].reset_index(drop=True)\n        X = scaler.fit_transform(df[feats]) if fit_scaler else scaler.transform(df[feats])   # same call as the original\n        self.X = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))\n        rul = np.minimum(df[\'RUL\'].to_numpy(dtype=np.float64), df[\'_cap\'].to_numpy()).astype(np.float32)\n        starts, labels, ids = [], [], []\n        pos = 0\n        for (_, u, cy), g in df.groupby([\'_part\', \'unit\', \'cycle\'], sort=False):   # subsets share unit numbers\n            idx = g.index.to_numpy(); n = len(idx)\n            if n < W: continue\n            assert idx[-1] - idx[0] + 1 == n, \'rows of a cycle are not contiguous\'\n            for k, st in enumerate(range(0, n - W + 1, S)):\n                starts.append(idx[0] + st); labels.append(float(rul[idx[0] + st + W - 1])); ids.append((int(u), int(cy), k))\n        self.starts = np.asarray(starts, np.int64); self.labels = np.asarray(labels, np.float32); self.ids = np.asarray(ids, np.int64)\n\n\nclass LazyWindowDataset(Dataset):\n    def __init__(self, w: Windows):\n        self.X, self.st, self.y = w.X, torch.from_numpy(w.starts), torch.from_numpy(w.labels).reshape(-1, 1)\n    def __len__(self): return int(self.y.shape[0])\n    def __getitem__(self, i):\n        s = int(self.st[i]); return self.X[s:s + W], self.y[i]\n\n\ndef loader(w, batch_size, shuffled, pin, num_workers=2):\n    kw = dict(dataset=LazyWindowDataset(w), batch_size=batch_size, shuffle=shuffled, drop_last=False, num_workers=num_workers, pin_memory=pin)\n    if num_workers > 0:\n        kw.update(prefetch_factor=2, persistent_workers=True, worker_init_fn=cbload.Data_Loader.seed_worker)\n    return DataLoader(**kw)\n\n\ndef train_predict(cbmain, args, train_w, test_w, device, epochs=None):\n    """Same order and loop as the original main()/Processor.train(); test windows are predicted once at the end."""\n    cbmain.set_seed(args.seed)\n    args.device = torch.device(device); args.model = cbmain.build_model(args)\n    args.optimizer = cbmain.build_optimizer(args); args.scheduler = cbmain.build_scheduler(args)\n    pin = args.device.type == \'cuda\'\n    tr = loader(train_w, args.train_batch_size, True, pin, args.num_workers)\n    te = loader(test_w, args.test_batch_size, False, pin, args.num_workers)\n    model = args.model.to(args.device); crit = torch.nn.MSELoss(); t0 = time.time(); t_ep = []\n    for ep in range(epochs or args.num_epochs):\n        model.train(); te0 = time.time()\n        for xb, yb in tr:\n            xb = xb.to(args.device, non_blocking=True); yb = yb.to(args.device, non_blocking=True)\n            loss = crit(model(xb), yb)\n            loss.backward(); args.optimizer.step(); args.optimizer.zero_grad(set_to_none=True)\n        args.scheduler.step(); t_ep.append(time.time() - te0)\n    model.eval(); out = []\n    with torch.no_grad():\n        for xb, _ in te: out.append(model(xb.to(args.device)).float().cpu().numpy().reshape(-1))\n    return np.concatenate(out), dict(sec_epochs=t_ep, sec_total=time.time() - t0, n_train=len(train_w.labels))\n\n\ndef cycle_stats(pred, test_w, cap):\n    """Per-cycle sufficient statistics of the clipped predictions (window-level errors can be rebuilt exactly)."""\n    p = np.clip(pred, 0, cap)\n    d = pd.DataFrame({\'cycle\': test_w.ids[:, 1], \'p\': p, \'p2\': p ** 2, \'raw\': pred, \'label\': test_w.labels})\n    g = d.groupby(\'cycle\').agg(n=(\'p\', \'size\'), s1=(\'p\', \'sum\'), s2=(\'p2\', \'sum\'), sraw=(\'raw\', \'sum\'), label=(\'label\', \'first\')).reset_index()\n    return g\n'
open(os.path.join(LOCAL, 'ws1gpu.py'), 'w').write(GLIB_SRC)
for p in [CB, LOCAL]:
    if p not in sys.path: sys.path.insert(0, p)
import importlib, main as cbmain, data_load as cbload, ws1gpu as G; importlib.reload(G)
MASK = os.environ.get('CB_MASK', f'{CB}/dataset/cruise_mask/common_altitude_cruise_mask.h5')
def cb_args(subset, model, seed):
    argv = ['--data_folder', FLAT, '--dataset_choice', str(SUB9.index(subset)), '--cruise_mask_path', MASK, '--window_size', '256', '--stride', '10',
            '--max_rul', str(CAPS[subset]), '--model_name', model, '--all_channels', '18', '--pred_len', '1', '--num_epochs', str(EPOCHS),
            '--train_batch_size', '512', '--test_batch_size', '512', '--learning_rate', '3e-4', '--seed', str(seed), '--optimizer_name', 'Adam',
            '--scheduler_name', 'None', '--device', DEV, '--run_name', 'ws1', '--logging_dir', os.path.join(LOCAL, 'logs')]
    old = sys.argv; sys.argv = ['main.py'] + argv
    try: return cbmain.parse_args()
    finally: sys.argv = old
json.dump(dict(cruisebench=CB_HEAD, ws1gpu=hashlib.sha256(GLIB_SRC.encode()).hexdigest()), open(f'{OUT}/ws1_dl_code.json', 'w'), indent=1)
print('CruiseBench', CB_HEAD[:7])

In [ ]:
# === 4. 추출 목록 (CPU 노트북과 같은 것인지 해시로 확인) ===
H1 = json.load(open(f'{W1}/sha256_ws1_cpu.json'))
assert sha(f'{W1}/ws1_draws.csv') == H1['ws1_draws.csv'], '추출 목록이 CPU 노트북 실행 당시와 다름'
DR = pd.read_csv(f'{W1}/ws1_draws.csv', dtype={'m': str}); WIN = pd.read_csv(f'{W1}/ws1_dev_windows.csv')
EOL = {}
for s in SUB9:
    r = pd.read_csv(f'{W1}/raw/{s}.csv.gz', usecols=['heldout', 'cycle', 'true']).drop_duplicates(['heldout'])
    for x, cy, t in r.itertuples(index=False): EOL[(s, int(x))] = int(cy + t)
DEVCACHE = {}
def dev(s):
    if s not in DEVCACHE:
        f = os.path.join(LOCAL, f'dev_{s}.pkl')
        if os.path.exists(f): DEVCACHE[s] = pd.read_pickle(f)
        else:
            df, feats = G.load_dev(f'{FLAT}/N-CMAPSS_{s}.h5', MASK); DEVCACHE[s] = (df, feats); pd.to_pickle((df, feats), f)
    return DEVCACHE[s]
print('추출', len(DR), '| 평가 엔진', DR.groupby(['subset', 'heldout']).ngroups)

In [ ]:
# === 5. 한 번의 학습 ===
from sklearn import preprocessing
def fit_name(scen, model, s, x, m, d): return f'{scen}_{model}_{s}_{x}_m{m}_d{d}.csv.gz'
def run_fit(scen, model, job, fits_dir, epochs=None):
    s, x, m, d = job['subset'], int(job['heldout']), str(job['m']), int(job['draw'])
    f = os.path.join(fits_dir, fit_name(scen, model, s, x, m, d))
    if os.path.exists(f): return None
    train = [int(u) for u in str(job['train']).split()]
    df, feats = dev(s)
    parts = [(df[df.unit.isin(train)], CAPS[s])]
    if scen == 'pool':
        parts = [(dev(o)[0], CAPS[o]) for o in SUB9 if o != s] + parts       # 다른 8개 서브셋의 개발 엔진 전체 + 대상 m대
    sc = preprocessing.MinMaxScaler()
    trw = G.Windows(parts, feats, sc, fit_scaler=True)
    tew = G.Windows([(df[df.unit == x], CAPS[s])], feats, sc, fit_scaler=False)
    args = cb_args(s, model, seed=d + 1)
    pred, info = G.train_predict(cbmain, args, trw, tew, DEV, epochs=epochs)
    st = G.cycle_stats(pred, tew, CAPS[s]).assign(scen=scen, model=model, subset=s, heldout=x, m=m, draw=d)
    st.to_csv(f + '.part', index=False, compression='gzip'); os.replace(f + '.part', f)
    return info

In [ ]:
# === 6. 점검 실행 (SMOKE = True일 때만; 성능 지표는 보지 않음) ===
if SMOKE:
    SM = os.path.join(LOCAL, 'smoke'); shutil.rmtree(SM, ignore_errors=True); os.makedirs(SM)
    s0 = 'DS08c-008'
    # (1) 창이 원 Data_Loader와 같은지: dev 엔진 전체, 정규화도 같게
    orig = cbload.Data_Loader(data_path=f'{FLAT}/N-CMAPSS_{s0}.h5', window_size=256, stride=10, max_rul=CAPS[s0], cruise_mask_path=MASK, num_workers=0)
    df, feats = dev(s0); ours = G.Windows([(df, CAPS[s0])], feats, preprocessing.MinMaxScaler(), fit_scaler=True)
    assert len(ours.labels) == len(orig.train_label) and np.array_equal(ours.labels, orig.train_label), '레이블이 원 코드와 다름'
    ds = G.LazyWindowDataset(ours); rng = np.random.default_rng(0)
    for i in list(range(200)) + list(rng.integers(0, len(ds), 500)):
        assert np.array_equal(ds[i][0].numpy(), orig.train_sample[i]), f'창 {i}가 원 코드와 다름'
    del orig; print('(1) 창과 레이블이 원 Data_Loader와 같음:', len(ours.labels), '창')
    # (2) 학습 2 epoch, (3) 이어서 실행(끝난 것은 건너뜀), (4) 시간 측정
    jobs = DR[(DR.subset == s0)].sort_values(['heldout', 'm', 'draw']).to_dict('records')
    jall = [j for j in jobs if j['m'] == 'all'][0]; T = {}
    for model in ['TSMixer', 'TCN']:
        info = run_fit('target', model, jall, SM, epochs=2); T[model] = info
    assert run_fit('target', 'TSMixer', jall, SM, epochs=2) is None, '끝난 학습을 다시 돌림'
    pj = [j for j in jobs if j['m'] == '2' and j['draw'] == 0][0]
    T['pool'] = run_fit('pool', 'TSMixer', pj, SM, epochs=1)
    print('(2)(3) 2 epoch 학습, 이어서 실행 확인 |', {k: dict(n_train=v['n_train'], sec_epoch=round(v['sec_epochs'][-1], 1)) for k, v in T.items()})
    json.dump(dict(gpu=GPU, timing={k: dict(n_train=v['n_train'], sec_epochs=v['sec_epochs'], sec_total=v['sec_total']) for k, v in T.items()}),
              open(f'{OUT}/ws1_dl_smoke_timing.json', 'w'), indent=1)
    FITS_DIR = SM
else:
    FITS_DIR = f'{OUT}/fits'

In [ ]:
# === 7. 계산량 규칙: 점검 실행의 학습 속도로 전체 시간 추정 (시간만 사용) ===
DEC = f'{OUT}/ws1_dl_compute_decision.json'
TM = json.load(open(f'{OUT}/ws1_dl_smoke_timing.json'))
def rate(k):                                   # 1 epoch, 창 1개당 초 (첫 epoch 제외 가능하면 제외)
    t = TM['timing'][k]; e = t['sec_epochs'][1:] or t['sec_epochs']; return float(np.mean(e)) / t['n_train']
nwin = WIN.groupby(['subset', 'unit']).n_win.sum()
def n_train(job, scen):
    tr = [int(u) for u in str(job['train']).split()]; n = sum(nwin.get((job['subset'], u), 0) for u in tr)
    if scen == 'pool': n += sum(nwin[o].sum() for o in SUB9 if o != job['subset'])
    return n
POOL_ALL = DR[(DR.m == '2') & (DR.draw == 0)]
POOL_27 = POOL_ALL[POOL_ALL.heldout.isin(POOL_ALL.groupby('subset').heldout.apply(lambda v: sorted(v)[:3]).explode().astype(int))]
def plan_hours(pool_jobs, draws, drop_m3):
    J = DR[DR.draw < draws]; J = J[J.m != '3'] if drop_m3 else J
    h = sum(rate(md) * EPOCHS * n_train(j, 'target') for md in ['TSMixer', 'TCN'] for j in J.to_dict('records'))
    h += sum(rate('pool') * EPOCHS * n_train(j, 'pool') for j in pool_jobs.to_dict('records'))
    return h / 3600, len(J), len(pool_jobs)
PLANS = [('full', POOL_ALL, 3, False), ('pool27', POOL_27, 3, False), ('draws2', POOL_27, 2, False), ('no_m3', POOL_27, 2, True)]
EST = {name: plan_hours(p, d, m3) for name, p, d, m3 in PLANS}
pick = next((n for n, p, d, m3 in PLANS if EST[n][0] <= BUDGET_H), PLANS[-1][0])
dec = dict(gpu=TM['gpu'], sec_per_window_epoch={k: rate(k) for k in ['TSMixer', 'TCN', 'pool']},
           estimates={n: dict(gpu_h=round(v[0], 1), target_fits_per_model=v[1], pool_fits=v[2]) for n, v in EST.items()},
           plan=pick, over_budget=bool(EST[pick][0] > BUDGET_H), rule='v1.2 2.6: >40 GPU-h -> pool 27, draws 2, drop m=3 (timing only)')
if not SMOKE and not os.path.exists(DEC): json.dump(dec, open(DEC, 'w'), indent=1)
dec = json.load(open(DEC)) if os.path.exists(DEC) else dec
print(json.dumps(dec, indent=1)); print(f"선택 {dec['plan']}: 약 {dec['estimates'][dec['plan']]['gpu_h']} GPU시간 ≈ {dec['estimates'][dec['plan']]['gpu_h'] * CU_PER_H:.0f} 단위 (L4)")

In [ ]:
# === 8. 본 실행 (끊기면 다시 실행: 끝난 학습은 건너뜀) ===
P_NAME = dec['plan']; _, POOL_J, DRAWS_N, DROP3 = [p for p in PLANS if p[0] == P_NAME][0]
TJ = DR[DR.draw < DRAWS_N]; TJ = TJ[TJ.m != '3'] if DROP3 else TJ
RUNLOG = f'{OUT}/ws1_dl_run_log.csv'
if not SMOKE:
    todo = [('target', 'TSMixer', j) for j in TJ.to_dict('records')] + [('target', 'TCN', j) for j in TJ.to_dict('records')] + \
           [('pool', 'TSMixer', j) for j in POOL_J.to_dict('records')]
    t0 = time.time(); k = 0
    for scen, model, j in todo:
        info = run_fit(scen, model, j, FITS_DIR)
        if info is None: continue
        k += 1
        pd.DataFrame([dict(scen=scen, model=model, subset=j['subset'], heldout=j['heldout'], m=j['m'], draw=j['draw'], n_train=info['n_train'],
                           minutes=info['sec_total'] / 60, gpu=GPU, finished=time.strftime('%Y-%m-%d %H:%M:%S'))]).to_csv(
            RUNLOG, mode='a', header=not os.path.exists(RUNLOG), index=False)
        if k % 10 == 0: print(f'{k}회 끝 | {scen} {model} {j["subset"]} | 누적 {(time.time() - t0) / 3600:.2f} h')
    print('본 실행 완료:', len(glob.glob(f'{FITS_DIR}/*.csv.gz')), '/', len(todo))

In [ ]:
# === 9. 엔진별 점수 (CPU 노트북과 같은 창, 같은 규칙: 창이 있는 첫 사이클 제외) ===
recs = []
for f in sorted(glob.glob(f'{FITS_DIR}/*.csv.gz')):
    g = pd.read_csv(f, dtype={'m': str}); s, x = g.subset.iloc[0], int(g.heldout.iloc[0]); eol = EOL[(s, x)]
    w = WIN[(WIN.subset == s) & (WIN.unit == x) & (WIN.cycle <= eol)]
    g = g.merge(w[['cycle', 'n_win']], on='cycle', how='left'); assert (g.n == g.n_win).all(), f'{f}: 창 수가 마스크와 다름'
    g = g[g.cycle > g.cycle.min()]
    y = np.minimum(eol - g.cycle.values, CAPS[s]).astype(float); assert np.allclose(y, g.label.values), f'{f}: 레이블 불일치'
    sse = float(np.sum(g.s2 - 2 * y * g.s1 + g.n * y ** 2))
    cyc = float(np.sum((g.sraw / g.n - (eol - g.cycle)) ** 2))       # 보조: 사이클 평균 예측, 상한 없음 (창이 있는 사이클만)
    recs.append(dict(scen=g.scen.iloc[0], model=g.model.iloc[0], subset=s, heldout=x, m=g.m.iloc[0], draw=int(g.draw.iloc[0]),
                     n=float(g.n.sum()), sse=sse, cyc_sse=cyc, n_cyc=len(g)))
DLD = pd.DataFrame(recs)
if SMOKE:
    print('(4) 채점 셀 작동 확인:', len(DLD), '개 (점수는 출력하지 않음)')
    shutil.rmtree(SM, ignore_errors=True)
    print('점검 통과. 점검 결과 파일은 지웠고 시간 측정(ws1_dl_smoke_timing.json)만 남겼습니다. 이 노트북을 커밋한 뒤 SMOKE = False로 본 실행하세요.')
else:
    DLD.to_csv(f'{OUT}/ws1_dl_draw_scores.csv', index=False)
    ENGD = DLD.groupby(['scen', 'model', 'subset', 'heldout', 'm'])[['n', 'sse', 'cyc_sse', 'n_cyc']].mean().reset_index()
    ENGD.to_csv(f'{OUT}/ws1_dl_engine_scores.csv', index=False)
    files = [p for p in sorted(glob.glob(f'{OUT}/**/*.*', recursive=True)) if not p.endswith('sha256_ws1_dl.json')]
    json.dump(dict(addendum=ADDENDUM_V12_URL, run=RUN_COMMIT_URL, cruisebench=CB_HEAD, gpu=GPU, files={os.path.relpath(p, OUT): sha(p) for p in files}),
              open(f'{OUT}/sha256_ws1_dl.json', 'w'), indent=1)
    print('엔진별 점수', len(ENGD), '행 저장 | 해시 기록. 비교와 붓스트랩은 WS1 채점 노트북에서 합니다.')